In [ ]:
import re
import time
from pathlib import Path

import anthropic
import pandas as pd
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
from dotenv import load_dotenv

project_root = Path.cwd().parent
data_dir = project_root / 'data'
reports_dir = project_root / 'reports'
load_dotenv(dotenv_path=data_dir.parent / ".env")

client = anthropic.Anthropic()

FEATURE_MODEL = "claude-haiku-4-5"  # cheap classification/counting task, doesn't need judge-tier quality
FEATURE_MAX_TOKENS = 150
LENGTH_CONFOUND_THRESHOLD = 0.7

df = pd.read_parquet(data_dir / "query_dataset.parquet")
print(f"Loaded {len(df)} labeled queries from Phase 1")

In [ ]:
# Free (non-LLM) features: length metrics + simple regex-based entity-density proxies.
# Note: this uses digit-count and mid-sentence-capitalized-word-count as a lightweight,
# documented proxy for numeric/named-entity density, not a full NER model -- out of scope
# for this project's timeline. Every feature here gets checked against raw query length
# below before being trusted (the length-confound trap from the prior project).
_DIGIT_RE = re.compile(r"\d+")
_CAPITALIZED_WORD_RE = re.compile(r"(?<!^)(?<!\. )(?<!\.\n)\b[A-Z][a-z]+")
_SENTENCE_SPLIT_RE = re.compile(r"[.!?]+")

def build_length_features(df):
    out = pd.DataFrame(index=df.index)
    out["query_char_count"] = df["query"].str.len()
    out["query_sentence_count"] = df["query"].apply(
        lambda q: max(1, len([s for s in _SENTENCE_SPLIT_RE.split(q) if s.strip()]))
    )
    out["query_avg_word_length"] = df["query"].apply(
        lambda q: (sum(len(w) for w in q.split()) / len(q.split())) if q.split() else 0.0
    )
    out["query_has_question_mark"] = df["query"].str.contains(r"\?", regex=True).astype(int)
    return out

def build_entity_density_features(df):
    out = pd.DataFrame(index=df.index)
    word_counts = df["query"].str.split().str.len().clip(lower=1)
    out["query_number_count"] = df["query"].apply(lambda q: len(_DIGIT_RE.findall(q)))
    out["query_number_density"] = out["query_number_count"] / word_counts
    out["query_capitalized_word_count"] = df["query"].apply(lambda q: len(_CAPITALIZED_WORD_RE.findall(q)))
    out["query_capitalized_density"] = out["query_capitalized_word_count"] / word_counts
    return out

length_feats = build_length_features(df)
entity_feats = build_entity_density_features(df)
free_feats = pd.concat([df[["query_id"]], length_feats, entity_feats], axis=1)
free_feats.to_parquet(data_dir / "phase2_free_features.parquet", index=False)
print(f"Built {len(free_feats.columns) - 1} free features for {len(free_feats)} queries.")
free_feats.describe()

In [ ]:
# LLM-extracted features: reasoning-step count, ambiguity score, domain, question type.
# One combined call per query (900 total, not per-model -- these are query-only features)
# via Haiku 4.5, thinking disabled (same fix needed in Phase 1 -- Sonnet/Opus 5 think by
# default and can exhaust max_tokens before writing any answer).
FEATURE_PROMPT_TEMPLATE = """Analyze this query and classify it along four dimensions. Do not answer the query itself.

Query:
{query}

1. reasoning_step_count: estimate how many distinct reasoning/lookup steps are needed to answer this well (integer, 1-10).
2. ambiguity_score: how underspecified/ambiguous is this query on its own, with no other context (integer 1-10; 1 = fully unambiguous, 10 = severely underspecified).
3. domain: the primary subject domain -- one of: coding, math, writing, factual, creative, other.
4. question_type: the shape of the task -- one of: single_fact_lookup, multi_step_reasoning, open_ended."""

FEATURE_SCHEMA = {
    "type": "object",
    "properties": {
        "reasoning_step_count": {"type": "integer"},
        "ambiguity_score": {"type": "integer"},
        "domain": {"type": "string", "enum": ["coding", "math", "writing", "factual", "creative", "other"]},
        "question_type": {"type": "string", "enum": ["single_fact_lookup", "multi_step_reasoning", "open_ended"]},
    },
    "required": ["reasoning_step_count", "ambiguity_score", "domain", "question_type"],
    "additionalProperties": False,
}

def build_llm_feature_batch_requests(df):
    requests = []
    for row in df.itertuples():
        prompt = FEATURE_PROMPT_TEMPLATE.format(query=row.query)
        requests.append(
            Request(
                custom_id=row.query_id,
                params=MessageCreateParamsNonStreaming(
                    model=FEATURE_MODEL,
                    max_tokens=FEATURE_MAX_TOKENS,
                    thinking={"type": "disabled"},
                    messages=[{"role": "user", "content": prompt}],
                    output_config={"format": {"type": "json_schema", "schema": FEATURE_SCHEMA}},
                ),
            )
        )
    return requests

feature_batch = client.messages.batches.create(requests=build_llm_feature_batch_requests(df))
print(f"Submitted feature batch {feature_batch.id}, status={feature_batch.processing_status}")

In [ ]:
def wait_for_batch(batch_id, poll_seconds=45):
    while True:
        b = client.messages.batches.retrieve(batch_id)
        if b.processing_status == "ended":
            return b
        print(f"status={b.processing_status}, request_counts={b.request_counts}")
        time.sleep(poll_seconds)

feature_batch = wait_for_batch(feature_batch.id)
print(f"Feature batch ended. request_counts={feature_batch.request_counts}")

In [ ]:
# Parse results. "succeeded but no text" is treated as a real failure, not a silent null.
import json

def extract_text(result):
    return next((b.text for b in result.result.message.content if b.type == "text"), None)

rows = []
failed = []
for result in client.messages.batches.results(feature_batch.id):
    query_id = result.custom_id
    text = extract_text(result) if result.result.type == "succeeded" else None
    if text is not None:
        parsed = json.loads(text)
        rows.append({
            "query_id": query_id,
            "reasoning_step_count": parsed.get("reasoning_step_count"),
            "ambiguity_score": parsed.get("ambiguity_score"),
            "domain": parsed.get("domain"),
            "question_type": parsed.get("question_type"),
        })
    else:
        rows.append({"query_id": query_id, "reasoning_step_count": None, "ambiguity_score": None, "domain": None, "question_type": None})
        error = result.result.type if result.result.type != "succeeded" else f"empty_text:{result.result.message.stop_reason}"
        failed.append({"query_id": query_id, "error": error})

llm_feats = pd.DataFrame(rows)
llm_feats.to_parquet(data_dir / "phase2_llm_features_raw.parquet", index=False)
print(f"Saved {len(llm_feats)} LLM-derived feature rows. {len(failed)} failed.")
llm_feats[["domain", "question_type"]].apply(lambda c: c.value_counts())

In [ ]:
# Length confound check -- the key carried-forward lesson from the prior project.
# Any candidate feature that just re-encodes raw query length gets dropped.
combined = df[["query_id", "query", "query_word_count", "source_category", "label"]].merge(
    free_feats, on="query_id"
).merge(llm_feats, on="query_id")

def length_confound_check(features_df, length_col="query_word_count", threshold=LENGTH_CONFOUND_THRESHOLD):
    numeric_cols = [c for c in features_df.select_dtypes(include="number").columns if c != length_col and not c.endswith("_id")]
    rows = []
    for col in numeric_cols:
        corr = features_df[col].corr(features_df[length_col])
        rows.append({
            "feature": col,
            "corr_with_length": round(float(corr), 3) if pd.notna(corr) else None,
            "length_confounded": bool(abs(corr) >= threshold) if pd.notna(corr) else False,
        })
    return pd.DataFrame(rows).sort_values("corr_with_length", key=lambda s: s.abs(), ascending=False)

report = length_confound_check(combined)
reports_dir.mkdir(exist_ok=True)
report.to_csv(reports_dir / "phase2_length_confound_check.csv", index=False)
print(report.to_string(index=False))

In [ ]:
# Drop the length-confounded features (|corr| >= 0.7 against query_word_count):
# query_char_count, query_sentence_count, and the raw (non-density) versions of
# capitalized-word-count and number-count. Their density-normalized counterparts pass
# clean and are kept -- same underlying signal, without the length confound.
LENGTH_CONFOUNDED_COLS = ["query_char_count", "query_sentence_count", "query_capitalized_word_count", "query_number_count"]
final = combined.drop(columns=[c for c in LENGTH_CONFOUNDED_COLS if c in combined.columns])
final.to_parquet(data_dir / "features.parquet", index=False)

print(f"Saved data/features.parquet: {len(final)} rows, {len(final.columns)} columns")
print(final.columns.tolist())